# Contrôle qualité des numéros de téléphone — PM FINESS+

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.chargement import resoudre_tables, charger_telecoms, charger_ege, charger_pm
from src.tel_checker import analyser_telephones, marquer_doublons

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Chargement des PM FINESS+

Le téléphone n'est plus une colonne de la structure : il vient de la table
`Telecom`, atteinte par la liaison contact et pivotée par canal.

In [2]:
TABLES = resoudre_tables(conn)

print("\nTelecoms :")
telecoms = charger_telecoms(conn, TABLES)

print("\nPM :")
df_src = charger_pm(conn, TABLES, telecoms)
print(f"\nPM FINESS actifs chargés : {len(df_src):,}")

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   telecom      -> T_FAC_Telecom
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact

Telecoms :
   3 lignes de canal sentinelle ecartees
   324 doublons (contact, canal) : premiere valeur conservee
   telecoms pivotes : 355,904 contacts, colonnes ['TEL_MAIL', 'TEL_TELECOPIE', 'TEL_TELEPHONE']

PM :
   84,015 lignes brutes
   84,015 lignes -> 56,194 structures apres deduplication

PM FINESS actifs chargés : 56,194


## 3. Vue d'ensemble

In [3]:
total     = len(df_src)
renseigne = df_src["TEL_TELEPHONE"].apply(
    lambda x: str(x).strip() not in ("", "nan", "None", "<NA>")).sum()
absent    = total - renseigne

print("─" * 52)
print("PANORAMA DES TÉLÉPHONES PM FINESS")
print("─" * 52)
print(f"  PM actifs total            : {total:>10,}")
print(f"  Téléphone renseigné        : {renseigne:>10,}  ({renseigne/total*100:.1f} %)")
print(f"  Téléphone absent           : {absent:>10,}  ({absent/total*100:.1f} %)")
print("─" * 52)

────────────────────────────────────────────────────
PANORAMA DES TÉLÉPHONES PM FINESS
────────────────────────────────────────────────────
  PM actifs total            :     56,194
  Téléphone renseigné        :     44,356  (78.9 %)
  Téléphone absent           :     11,838  (21.1 %)
────────────────────────────────────────────────────


## 4. Nettoyage, classification et doublons

In [4]:
print("Nettoyage + classification + doublons...")
df = analyser_telephones(df_src, col_tel="TEL_TELEPHONE", col_commune="TX_CogCommune")
df = marquer_doublons(df, col_tel_clean="telephone_clean", col_id="NB_PmSmsseId")
print(f"Analyse terminée — {len(df):,} PM traités.")
df[["TX_DenominationPm", "TEL_TELEPHONE", "telephone_clean", "categorie",
    "motif", "zone_numero", "zone_attendue"]].head(15)

Nettoyage + classification + doublons...
Analyse terminée — 56,194 PM traités.


,TX_DenominationPm,TEL_TELEPHONE,telephone_clean,categorie,motif,zone_numero,zone_attendue
0,CLINIQUE CONVERT,0428631234,0428631234,VALIDE,,04,04
1,CLINIQUE DU SOUFFLE LE PONTET,0474404949,0474404949,VALIDE,,04,04
2,ASS.ACCUEIL FORMA INSERT PERSON SOURD,0474226685,0474226685,VALIDE,,04,04
3,RESIDENCE FONTELUNE,0474461420,0474461420,VALIDE,,04,04
4,LA RESIDENCE D'URFE,0385301285,0385301285,INCOHERENCE_GEO,"numéro zone 03, commune zone 04",03,04
5,EHPAD L'ALBIZIA,0474399662,0474399662,VALIDE,,04,04
6,EHPAD FONDATION COSTAZ CHAMPAGNE,0479876141,0479876141,VALIDE,,04,04
7,MAISON DE RETRAITE DE COLIGNY,0474301056,0474301056,VALIDE,,04,04
8,MAISON DE RETRAITE DE LAGNIEU,0474357200,0474357200,VALIDE,,04,04
9,MAISON DE RETRAITE DE MONTLUEL,0478060243,0478060243,VALIDE,,04,04


## 5. Distribution des catégories

In [5]:
ordre = ["MANQUANT", "ANOMALIE_CRITIQUE", "SURTAXE", "MOBILE",
         "INCOHERENCE_GEO", "DOUBLON", "VALIDE"]
labels = {
    "MANQUANT": "Manquant (absent / faux vide)",
    "ANOMALIE_CRITIQUE": "[1] Anomalie critique",
    "SURTAXE": "[2] Surtaxe (08 payant)",
    "MOBILE": "[2] Mobile (06/07)",
    "INCOHERENCE_GEO": "[2] Incohérence géographique",
    "DOUBLON": "[2] Doublon",
    "VALIDE": "[0] Valide",
}
vc = df["categorie"].value_counts()

print("─" * 60)
print("DISTRIBUTION DES CATÉGORIES")
print("─" * 60)
for cat in ordre:
    n = int(vc.get(cat, 0))
    print(f"  {labels[cat]:<34} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 60)

────────────────────────────────────────────────────────────
DISTRIBUTION DES CATÉGORIES
────────────────────────────────────────────────────────────
  Manquant (absent / faux vide)      :   11,838  ( 21.1 %)
  [1] Anomalie critique              :       26  (  0.0 %)
  [2] Surtaxe (08 payant)            :       71  (  0.1 %)
  [2] Mobile (06/07)                 :    1,979  (  3.5 %)
  [2] Incohérence géographique       :       80  (  0.1 %)
  [2] Doublon                        :    2,743  (  4.9 %)
  [0] Valide                         :   39,457  ( 70.2 %)
────────────────────────────────────────────────────────────


## 6. Anomalies critiques — détail par motif

In [6]:
df_crit = df[df["categorie"] == "ANOMALIE_CRITIQUE"].copy()
print(f"Total anomalies critiques : {len(df_crit):,}\n")

COLS = ["TX_NumFinessPm", "TX_DenominationPm", "TEL_TELEPHONE", "telephone_clean", "motif"]
cols = [c for c in COLS if c in df_crit.columns]

for motif, groupe in df_crit.groupby("motif"):
    print(f"\n{'─'*60}")
    print(f"  {motif}  ({len(groupe):,} cas)")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

Total anomalies critiques : 26


────────────────────────────────────────────────────────────
  longueur 11 != 10  (8 cas)
────────────────────────────────────────────────────────────
TX_NumFinessPm      TX_DenominationPm TEL_TELEPHONE telephone_clean             motif
     120002332 PHARMACIE SARAH CHEKIR   00565443110     00565443110 longueur 11 != 10
     220003792     PHARMACIE  PERETTI   02996208134     02996208134 longueur 11 != 10
     270021587   PHARMACIE DU PLATEAU   00232373167     00232373167 longueur 11 != 10
     440011450       PHARMACIE LIBEAU   02409109053     02409109053 longueur 11 != 10
     600006274  SELARL PHARMACIE AGEL   00344021582     00344021582 longueur 11 != 10

────────────────────────────────────────────────────────────
  préfixe invalide (00)  (3 cas)
────────────────────────────────────────────────────────────
TX_NumFinessPm                 TX_DenominationPm TEL_TELEPHONE telephone_clean                 motif
     240018341 CC ISLE VERN SALEMBRE EN PER

## 7. Signaux qualité — surtaxes, mobiles, incohérences géo, doublons

In [7]:
for cat, titre in [("SURTAXE", "SURTAXES — 08 payant (signal fort)"),
                   ("MOBILE", "MOBILES — 06/07 (signal faible)"),
                   ("INCOHERENCE_GEO", "INCOHÉRENCES GÉO (informatif — portabilité depuis 2023)"),
                   ("DOUBLON", "DOUBLONS (informatif — standard mutualisé possible)")]:
    g = df[df["categorie"] == cat]
    print(f"\n{'─'*60}")
    print(f"  {titre}  ({len(g):,} cas)")
    print(f"{'─'*60}")
    if len(g):
        print(g[cols].head(5).to_string(index=False))


────────────────────────────────────────────────────────────
  SURTAXES — 08 payant (signal fort)  (71 cas)
────────────────────────────────────────────────────────────
TX_NumFinessPm                 TX_DenominationPm TEL_TELEPHONE telephone_clean           motif
     060000221 CLINIQUE INTERNATIONALE DE CANNES    0826200210      0826200210 08 payant (082)
     060004959         CLINIQUE DU PARC IMPERIAL    0826307576      0826307576 08 payant (082)
     100010792       GCS HOPITAL PRIVE DE L'AUBE    0820151111      0820151111 08 payant (082)
     120787064           ASSOCIATION PIERREFICHE    0890707425      0890707425 08 payant (089)
     130000813             CLINIQUE DE LA CIOTAT    0826207580      0826207580 08 payant (082)

────────────────────────────────────────────────────────────
  MOBILES — 06/07 (signal faible)  (1,979 cas)
────────────────────────────────────────────────────────────
TX_NumFinessPm                TX_DenominationPm TEL_TELEPHONE telephone_clean          mot

## 8. Synthèse finale

In [8]:
n_total = len(df)
compte = df["categorie"].value_counts()

print("=" * 60)
print("SYNTHÈSE GLOBALE — TÉLÉPHONES PM FINESS")
print("=" * 60)
print(f"  PM actifs analysés        : {n_total:>8,}")
for cat in ordre:
    print(f"  {labels[cat]:<34} : {int(compte.get(cat, 0)):>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — TÉLÉPHONES PM FINESS
  PM actifs analysés        :   56,194
  Manquant (absent / faux vide)      :   11,838
  [1] Anomalie critique              :       26
  [2] Surtaxe (08 payant)            :       71
  [2] Mobile (06/07)                 :    1,979
  [2] Incohérence géographique       :       80
  [2] Doublon                        :    2,743
  [0] Valide                         :   39,457


## 9. Export Excel — Rapport de signalement

In [9]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/tel")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_tel_pm.xlsx"

COLS_EXPORT = [
    "NB_PmSmsseId", "TX_NumFinessPm", "TX_DenominationPm",
    "TX_CogCommune", "departement", "adresse_complete",
    "TEL_TELEPHONE", "telephone_clean",
    "categorie", "motif", "zone_numero", "zone_attendue",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

df_synth = pd.DataFrame([
    {"Catégorie": labels[cat], "Nombre": int(compte.get(cat, 0)),
      "Part": f"{compte.get(cat, 0)/n_total*100:.1f} %"}
    for cat in ordre
] + [{"Catégorie": "TOTAL", "Nombre": n_total, "Part": "100 %"}])

FEUILLES = [
    ("Manquants",         "MANQUANT",          "757575", "EEEEEE"),
    ("Anomalies_Critiques", "ANOMALIE_CRITIQUE", "C62828", "F8CECC"),
    ("Surtaxes",          "SURTAXE",           "C62828", "F8CECC"),
    ("Mobiles",           "MOBILE",            "F57C00", "FFF2CC"),
    ("Incoherences_Geo",  "INCOHERENCE_GEO",   "F57C00", "FFF2CC"),
    ("Doublons",          "DOUBLON",           "F57C00", "FFF2CC"),
    ("Numeros_Valides",   "VALIDE",            "2E7D32", "EBF5EB"),
]

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom_f, categorie, ent, fond in FEUILLES:
        donnees = df[df["categorie"] == categorie][cols_dispo]
        if len(donnees) == 0:
            continue
        donnees.to_excel(writer, sheet_name=nom_f, index=False)
        style_entete(writer.sheets[nom_f], ent)
        remplir(writer.sheets[nom_f], fond)
        auto_width(writer.sheets[nom_f])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print("\nFeuilles produites :")
for nom_f, categorie, _, _ in FEUILLES:
    print(f"  {nom_f:<22} : {int(compte.get(categorie, 0)):,}")

Export OK → /home/jovyan/work/finess+_refonte/signalement_data_contact_finess+/results/tel/signalement_tel_pm.xlsx

Feuilles produites :
  Manquants              : 11,838
  Anomalies_Critiques    : 26
  Surtaxes               : 71
  Mobiles                : 1,979
  Incoherences_Geo       : 80
  Doublons               : 2,743
  Numeros_Valides        : 39,457
